In [1]:
import os
import numpy as np
import pandas as pd

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
data_folder = r"Data"

In [3]:
data = {}

for i in range(1, 9):
    file_path = os.path.join(
        data_folder,
        f"function_{i}.csv"
    )
    
    data[i] = pd.read_csv(file_path)

    print(
        f"Function {i}: "
        f"{len(data[i])} observations, "
        f"{len(data[i].columns) - 1} dimensions"
    )

Function 1: 14 observations, 2 dimensions
Function 2: 14 observations, 2 dimensions
Function 3: 19 observations, 3 dimensions
Function 4: 34 observations, 4 dimensions
Function 5: 24 observations, 4 dimensions
Function 6: 24 observations, 5 dimensions
Function 7: 34 observations, 6 dimensions
Function 8: 44 observations, 8 dimensions


In [4]:
week_5_inputs = {
    1: [0.943533, 0.359421],
    2: [0.713092, 0.138155],
    3: [0.384686, 0.478261, 0.416295],
    4: [0.462451, 0.342242, 0.380323, 0.445636],
    5: [0.938942, 0.989452, 0.984552, 0.971823],
    6: [0.248909, 0.369411, 0.484746, 0.674732, 0.011701],
    7: [0.132842, 0.294043, 0.285122, 0.050957, 0.281982, 0.619768],
    8: [0.034135, 0.275291, 0.032179, 0.508682,
        0.760754, 0.617950, 0.034127, 0.349420]
}

week_5_outputs = {
    1: -2.4467665937353882e-120,
    2: 0.7469622511600147,
    3: -0.016451292503935306,
    4: -0.6383428250895409,
    5: 6691.93167361189,
    6: -0.4568735372038142,
    7: 1.6447360459041676,
    8: 9.742277082966
}

In [5]:
previous_bests = {}

for i in range(1, 9):

    # Record best BEFORE adding new observation
    previous_bests[i] = data[i]["output"].max()

    input_columns = [
        col for col in data[i].columns
        if col != "output"
    ]

    new_row = dict(
        zip(
            input_columns,
            week_5_inputs[i]
        )
    )

    new_row["output"] = week_5_outputs[i]

    data[i] = pd.concat(
        [
            data[i],
            pd.DataFrame([new_row])
        ],
        ignore_index=True
    )

In [6]:
print("=" * 60)
print("WEEK 5 RESULTS")
print("=" * 60)

for i in range(1, 9):

    previous_best = previous_bests[i]
    new_output = week_5_outputs[i]

    difference = new_output - previous_best
    updated_best = data[i]["output"].max()

    improved = new_output > previous_best

    print("\n" + "=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(f"Previous best: {previous_best:.6f}")
    print(f"New output:    {new_output:.6f}")
    print(f"Difference:    {difference:.6f}")
    print(f"Updated best:  {updated_best:.6f}")

    if improved:
        print("Result: NEW BEST")
    else:
        print("Result: Did not improve")

WEEK 5 RESULTS

FUNCTION 1
Previous best: 0.000000
New output:    -0.000000
Difference:    -0.000000
Updated best:  0.000000
Result: Did not improve

FUNCTION 2
Previous best: 0.611205
New output:    0.746962
Difference:    0.135757
Updated best:  0.746962
Result: NEW BEST

FUNCTION 3
Previous best: -0.020167
New output:    -0.016451
Difference:    0.003716
Updated best:  -0.016451
Result: NEW BEST

FUNCTION 4
Previous best: -0.300645
New output:    -0.638343
Difference:    -0.337698
Updated best:  -0.300645
Result: Did not improve

FUNCTION 5
Previous best: 4857.449601
New output:    6691.931674
Difference:    1834.482072
Updated best:  6691.931674
Result: NEW BEST

FUNCTION 6
Previous best: -0.458511
New output:    -0.456874
Difference:    0.001637
Updated best:  -0.456874
Result: NEW BEST

FUNCTION 7
Previous best: 2.536939
New output:    1.644736
Difference:    -0.892203
Updated best:  2.536939
Result: Did not improve

FUNCTION 8
Previous best: 9.911249
New output:    9.742277
Diff

In [7]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.preprocessing import StandardScaler

gp_models = {}
gp_scalers = {}

for i in range(1, 9):

    X = (
        data[i]
        .drop(columns="output")
        .values
        .astype(float)
    )

    y = (
        data[i]["output"]
        .values
        .astype(float)
    )

    # Standardise outputs
    scaler = StandardScaler()
    y_scaled = scaler.fit_transform(
        y.reshape(-1, 1)
    ).ravel()

    # Fit GP
    kernel = RBF(
        length_scale=0.2,
        length_scale_bounds=(1e-5, 1e5)
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-6,
        normalize_y=False,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y_scaled)

    gp_models[i] = gp
    gp_scalers[i] = scaler

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)
    print(f"Fitted kernel: {gp.kernel_}")
    print()

FUNCTION 1
Fitted kernel: RBF(length_scale=1e-05)

FUNCTION 2
Fitted kernel: RBF(length_scale=0.0521)

FUNCTION 3
Fitted kernel: RBF(length_scale=0.185)

FUNCTION 4
Fitted kernel: RBF(length_scale=0.423)

FUNCTION 5
Fitted kernel: RBF(length_scale=0.389)

FUNCTION 6
Fitted kernel: RBF(length_scale=0.535)

FUNCTION 7
Fitted kernel: RBF(length_scale=0.248)

FUNCTION 8
Fitted kernel: RBF(length_scale=1)



In [8]:
# Generate candidate points and examine GP behaviour

candidate_points = {}
gp_diagnostics = {}

rng = np.random.default_rng(42)

for i in range(1, 9):

    n_dimensions = len(
        data[i].columns
    ) - 1

    # Generate 100,000 candidate points
    candidates = rng.random(
        (100000, n_dimensions)
    )

    candidate_points[i] = candidates

    # GP predictions in standardised output space
    mean_scaled, std_scaled = (
        gp_models[i].predict(
            candidates,
            return_std=True
        )
    )

    # Convert predictions back to original output scale
    scaler = gp_scalers[i]

    mean = scaler.inverse_transform(
        mean_scaled.reshape(-1, 1)
    ).ravel()

    std = (
        std_scaled *
        scaler.scale_[0]
    )

    gp_diagnostics[i] = {
        "mean": mean,
        "std": std
    }

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(
        f"Predicted mean minimum: "
        f"{mean.min():.10f}"
    )
    print(
        f"Predicted mean maximum: "
        f"{mean.max():.10f}"
    )
    print(
        f"Predicted mean range:   "
        f"{mean.max() - mean.min():.10f}"
    )

    print()

    print(
        f"Uncertainty minimum:    "
        f"{std.min():.10f}"
    )
    print(
        f"Uncertainty maximum:    "
        f"{std.max():.10f}"
    )
    print(
        f"Uncertainty range:      "
        f"{std.max() - std.min():.10f}"
    )

    print()

FUNCTION 1
Predicted mean minimum: -0.0006345829
Predicted mean maximum: -0.0000000358
Predicted mean range:   0.0006345470

Uncertainty minimum:    0.0000176862
Uncertainty maximum:    0.0016717864
Uncertainty range:      0.0016541003

FUNCTION 2
Predicted mean minimum: -0.0824904227
Predicted mean maximum: 0.7489262743
Predicted mean range:   0.8314166970

Uncertainty minimum:    0.0002595097
Uncertainty maximum:    0.2595015299
Uncertainty range:      0.2592420202

FUNCTION 3
Predicted mean minimum: -0.3976338817
Predicted mean maximum: -0.0085327376
Predicted mean range:   0.3891011442

Uncertainty minimum:    0.0000781193
Uncertainty maximum:    0.0781185128
Uncertainty range:      0.0780403935

FUNCTION 4
Predicted mean minimum: -34.6476933117
Predicted mean maximum: -0.2133622981
Predicted mean range:   34.4343310136

Uncertainty minimum:    0.0085059193
Uncertainty maximum:    7.3947778303
Uncertainty range:      7.3862719110

FUNCTION 5
Predicted mean minimum: -733.8178440534


In [9]:
from scipy.stats import norm

gp_candidate_results = {}

for i in range(1, 9):

    candidates = candidate_points[i]

    mean = gp_diagnostics[i]["mean"]
    std = gp_diagnostics[i]["std"]

    X_existing = (
        data[i]
        .drop(columns="output")
        .values
        .astype(float)
    )

    y_existing = (
        data[i]["output"]
        .values
        .astype(float)
    )

    # Current best
    best_idx = np.argmax(y_existing)
    current_best = y_existing[best_idx]
    best_point = X_existing[best_idx]

    # -------------------------
    # UCB
    # -------------------------

    kappa = 1.5
    ucb_scores = mean + kappa * std

    ucb_idx = np.argmax(ucb_scores)
    ucb_point = candidates[ucb_idx]

    # -------------------------
    # Expected Improvement
    # -------------------------

    improvement = mean - current_best

    with np.errstate(divide="ignore", invalid="ignore"):
        Z = np.divide(
            improvement,
            std,
            out=np.zeros_like(improvement),
            where=std > 0
        )

        ei_scores = (
            improvement * norm.cdf(Z)
            + std * norm.pdf(Z)
        )

    ei_scores[std <= 0] = 0

    ei_idx = np.argmax(ei_scores)
    ei_point = candidates[ei_idx]

    # -------------------------
    # Distances
    # -------------------------

    ucb_nearest = np.min(
        np.linalg.norm(
            X_existing - ucb_point,
            axis=1
        )
    )

    ei_nearest = np.min(
        np.linalg.norm(
            X_existing - ei_point,
            axis=1
        )
    )

    ucb_from_best = np.linalg.norm(
        ucb_point - best_point
    )

    ei_from_best = np.linalg.norm(
        ei_point - best_point
    )

    candidate_distance = np.linalg.norm(
        ucb_point - ei_point
    )

    # Save results
    gp_candidate_results[i] = {
        "ucb_point": ucb_point,
        "ucb_mean": mean[ucb_idx],
        "ucb_std": std[ucb_idx],
        "ucb_score": ucb_scores[ucb_idx],

        "ei_point": ei_point,
        "ei_mean": mean[ei_idx],
        "ei_std": std[ei_idx],
        "ei_score": ei_scores[ei_idx],

        "current_best": current_best,
        "best_point": best_point
    }

    # -------------------------
    # Print
    # -------------------------

    print("=" * 70)
    print(f"FUNCTION {i}")
    print("=" * 70)

    print("\nUCB (κ = 1.5)")
    print(np.round(ucb_point, 6))
    print(
        f"Predicted output:               "
        f"{mean[ucb_idx]:.6f}"
    )
    print(
        f"Uncertainty:                    "
        f"{std[ucb_idx]:.6f}"
    )
    print(
        f"UCB score:                      "
        f"{ucb_scores[ucb_idx]:.6f}"
    )
    print(
        f"Distance from current best:     "
        f"{ucb_from_best:.4f}"
    )
    print(
        f"Distance from nearest existing: "
        f"{ucb_nearest:.4f}"
    )

    print("\nEXPECTED IMPROVEMENT")
    print(np.round(ei_point, 6))
    print(
        f"Predicted output:               "
        f"{mean[ei_idx]:.6f}"
    )
    print(
        f"Uncertainty:                    "
        f"{std[ei_idx]:.6f}"
    )
    print(
        f"EI score:                       "
        f"{ei_scores[ei_idx]:.6f}"
    )
    print(
        f"Distance from current best:     "
        f"{ei_from_best:.4f}"
    )
    print(
        f"Distance from nearest existing: "
        f"{ei_nearest:.4f}"
    )

    print(
        f"\nDistance between UCB and EI candidates: "
        f"{candidate_distance:.4f}"
    )

    print()

FUNCTION 1

UCB (κ = 1.5)
[0.773956 0.438878]
Predicted output:               -0.000635
Uncertainty:                    0.001672
UCB score:                      0.001873
Distance from current best:     0.2972
Distance from nearest existing: 0.1807

EXPECTED IMPROVEMENT
[0.773956 0.438878]
Predicted output:               -0.000635
Uncertainty:                    0.001672
EI score:                       0.000397
Distance from current best:     0.2972
Distance from nearest existing: 0.1807

Distance between UCB and EI candidates: 0.0000

FUNCTION 2

UCB (κ = 1.5)
[0.732816 0.105087]
Predicted output:               0.644023
Uncertainty:                    0.166402
UCB score:                      0.893626
Distance from current best:     0.0385
Distance from nearest existing: 0.0385

EXPECTED IMPROVEMENT
[0.729513 0.111999]
Predicted output:               0.679240
Uncertainty:                    0.139045
EI score:                       0.028062
Distance from current best:     0.0309
Distance

In [10]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

class SmallNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)


nn_models = {}
nn_scalers = {}
nn_training_losses = {}

for i in range(1, 9):

    X = (
        data[i]
        .drop(columns="output")
        .values
        .astype(np.float32)
    )

    y = (
        data[i]["output"]
        .values
        .astype(np.float32)
        .reshape(-1, 1)
    )

    # Standardise outputs
    y_scaler = StandardScaler()

    y_scaled = (
        y_scaler
        .fit_transform(y)
        .astype(np.float32)
    )

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        y_scaled,
        dtype=torch.float32
    )

    models = []
    final_losses = []

    # Train five networks per function
    for seed in range(5):

        torch.manual_seed(seed)

        model = SmallNN(
            input_dim=X.shape[1]
        )

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.01
        )

        loss_function = nn.MSELoss()

        # Same training length as Week 5
        for epoch in range(2000):

            optimizer.zero_grad()

            predictions = model(
                X_tensor
            )

            loss = loss_function(
                predictions,
                y_tensor
            )

            loss.backward()
            optimizer.step()

        models.append(model)
        final_losses.append(
            loss.item()
        )

    nn_models[i] = models
    nn_scalers[i] = y_scaler
    nn_training_losses[i] = final_losses

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(
        f"Mean final training loss: "
        f"{np.mean(final_losses):.6f}"
    )

    print(
        f"Minimum final training loss: "
        f"{np.min(final_losses):.6f}"
    )

    print(
        f"Maximum final training loss: "
        f"{np.max(final_losses):.6f}"
    )

    print()

FUNCTION 1
Mean final training loss: 0.014672
Minimum final training loss: 0.000000
Maximum final training loss: 0.072590

FUNCTION 2
Mean final training loss: 0.023384
Minimum final training loss: 0.000267
Maximum final training loss: 0.087228

FUNCTION 3
Mean final training loss: 0.012198
Minimum final training loss: 0.000000
Maximum final training loss: 0.042261

FUNCTION 4
Mean final training loss: 0.001875
Minimum final training loss: 0.000256
Maximum final training loss: 0.005423

FUNCTION 5
Mean final training loss: 0.000085
Minimum final training loss: 0.000021
Maximum final training loss: 0.000164

FUNCTION 6
Mean final training loss: 0.003526
Minimum final training loss: 0.000011
Maximum final training loss: 0.009342

FUNCTION 7
Mean final training loss: 0.000907
Minimum final training loss: 0.000000
Maximum final training loss: 0.002452

FUNCTION 8
Mean final training loss: 0.001338
Minimum final training loss: 0.000133
Maximum final training loss: 0.003092



In [11]:
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error

nn_validation_results = {}

for i in range(1, 9):

    X = (
        data[i]
        .drop(columns="output")
        .values
        .astype(np.float32)
    )

    y = (
        data[i]["output"]
        .values
        .astype(np.float32)
    )

    loo = LeaveOneOut()

    predictions = []
    actuals = []
    baseline_predictions = []

    for train_idx, test_idx in loo.split(X):

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        # Standardise training outputs
        y_scaler = StandardScaler()

        y_train_scaled = (
            y_scaler
            .fit_transform(
                y_train.reshape(-1, 1)
            )
            .astype(np.float32)
        )

        X_train_tensor = torch.tensor(
            X_train,
            dtype=torch.float32
        )

        X_test_tensor = torch.tensor(
            X_test,
            dtype=torch.float32
        )

        y_train_tensor = torch.tensor(
            y_train_scaled,
            dtype=torch.float32
        )

        # Train one small NN for this fold
        torch.manual_seed(42)

        model = SmallNN(
            input_dim=X.shape[1]
        )

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.01
        )

        loss_function = nn.MSELoss()

        for epoch in range(1000):

            optimizer.zero_grad()

            pred = model(
                X_train_tensor
            )

            loss = loss_function(
                pred,
                y_train_tensor
            )

            loss.backward()
            optimizer.step()

        # Predict held-out observation
        model.eval()

        with torch.no_grad():

            pred_scaled = (
                model(X_test_tensor)
                .numpy()
            )

        pred_original = (
            y_scaler
            .inverse_transform(
                pred_scaled
            )
            .ravel()[0]
        )

        predictions.append(
            pred_original
        )

        actuals.append(
            y_test[0]
        )

        # Simple baseline:
        # predict training-set mean
        baseline_predictions.append(
            np.mean(y_train)
        )

    nn_mae = mean_absolute_error(
        actuals,
        predictions
    )

    baseline_mae = mean_absolute_error(
        actuals,
        baseline_predictions
    )

    beats_baseline = (
        nn_mae < baseline_mae
    )

    nn_validation_results[i] = {
        "nn_mae": nn_mae,
        "baseline_mae": baseline_mae,
        "beats_baseline": beats_baseline
    }

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(
        f"NN LOOCV MAE:       "
        f"{nn_mae:.6f}"
    )

    print(
        f"Mean baseline MAE:  "
        f"{baseline_mae:.6f}"
    )

    print(
        f"NN beats baseline:  "
        f"{beats_baseline}"
    )

    print()

FUNCTION 1
NN LOOCV MAE:       0.001675
Mean baseline MAE:  0.001179
NN beats baseline:  False

FUNCTION 2
NN LOOCV MAE:       0.267443
Mean baseline MAE:  0.241496
NN beats baseline:  False

FUNCTION 3
NN LOOCV MAE:       0.067227
Mean baseline MAE:  0.048537
NN beats baseline:  False

FUNCTION 4
NN LOOCV MAE:       1.827993
Mean baseline MAE:  6.878180
NN beats baseline:  True

FUNCTION 5
NN LOOCV MAE:       273.005550
Mean baseline MAE:  1375.410321
NN beats baseline:  True

FUNCTION 6
NN LOOCV MAE:       0.309082
Mean baseline MAE:  0.476405
NN beats baseline:  True

FUNCTION 7
NN LOOCV MAE:       0.323396
Mean baseline MAE:  0.507559
NN beats baseline:  True

FUNCTION 8
NN LOOCV MAE:       0.294854
Mean baseline MAE:  0.920553
NN beats baseline:  True



In [12]:
nn_candidate_results = {}

for i in range(1, 9):

    candidates = candidate_points[i].astype(
        np.float32
    )

    candidate_tensor = torch.tensor(
        candidates,
        dtype=torch.float32
    )

    models = nn_models[i]
    scaler = nn_scalers[i]

    model_predictions = []

    # Generate predictions from all five networks
    for model in models:

        model.eval()

        with torch.no_grad():

            pred_scaled = (
                model(candidate_tensor)
                .numpy()
            )

        pred_original = (
            scaler
            .inverse_transform(pred_scaled)
            .ravel()
        )

        model_predictions.append(
            pred_original
        )

    model_predictions = np.array(
        model_predictions
    )

    # Ensemble mean prediction
    mean_prediction = np.mean(
        model_predictions,
        axis=0
    )

    # Disagreement between networks
    model_disagreement = np.std(
        model_predictions,
        axis=0
    )

    # Select candidate with highest
    # ensemble mean prediction
    best_idx = np.argmax(
        mean_prediction
    )

    nn_point = candidates[best_idx]
    nn_predicted_output = (
        mean_prediction[best_idx]
    )
    nn_disagreement = (
        model_disagreement[best_idx]
    )

    # Existing observations
    X_existing = (
        data[i]
        .drop(columns="output")
        .values
        .astype(float)
    )

    y_existing = (
        data[i]["output"]
        .values
        .astype(float)
    )

    # Current best observation
    best_observation_idx = np.argmax(
        y_existing
    )

    best_point = X_existing[
        best_observation_idx
    ]

    # Distances
    distance_from_best = np.linalg.norm(
        nn_point - best_point
    )

    nearest_distance = np.min(
        np.linalg.norm(
            X_existing - nn_point,
            axis=1
        )
    )

    # Save results
    nn_candidate_results[i] = {
        "point": nn_point,
        "predicted_output": nn_predicted_output,
        "disagreement": nn_disagreement,
        "distance_from_best": distance_from_best,
        "nearest_distance": nearest_distance
    }

    # Print
    print("=" * 70)
    print(f"FUNCTION {i}")
    print("=" * 70)

    print("NN selected point:")
    print(
        np.round(
            nn_point,
            6
        )
    )

    print(
        f"NN predicted output:             "
        f"{nn_predicted_output:.6f}"
    )

    print(
        f"NN model disagreement:          "
        f"{nn_disagreement:.6f}"
    )

    print(
        f"Distance from current best:     "
        f"{distance_from_best:.4f}"
    )

    print(
        f"Distance from nearest existing: "
        f"{nearest_distance:.4f}"
    )

    print()

FUNCTION 1
NN selected point:
[9.98294e-01 7.09000e-04]
NN predicted output:             0.003677
NN model disagreement:          0.002351
Distance from current best:     0.7795
Distance from nearest existing: 0.3077

FUNCTION 2
NN selected point:
[0.998307 0.001142]
NN predicted output:             2.724940
NN model disagreement:          1.237037
Distance from current best:     0.3164
Distance from nearest existing: 0.3164

FUNCTION 3
NN selected point:
[0.364502 0.118478 0.645908]
NN predicted output:             -0.001139
NN model disagreement:          0.032445
Distance from current best:     0.4273
Distance from nearest existing: 0.2033

FUNCTION 4
NN selected point:
[0.453836 0.433425 0.386591 0.463858]
NN predicted output:             -0.374836
NN model disagreement:          0.459252
Distance from current best:     0.0619
Distance from nearest existing: 0.0619

FUNCTION 5
NN selected point:
[0.939869 0.992261 0.978815 0.999723]
NN predicted output:             6691.842285
NN m

In [13]:
model_comparison = {}

for i in range(1, 9):

    nn_point = nn_candidate_results[i]["point"]
    ucb_point = gp_candidate_results[i]["ucb_point"]
    ei_point = gp_candidate_results[i]["ei_point"]

    nn_to_ucb = np.linalg.norm(
        nn_point - ucb_point
    )

    nn_to_ei = np.linalg.norm(
        nn_point - ei_point
    )

    ucb_to_ei = np.linalg.norm(
        ucb_point - ei_point
    )

    model_comparison[i] = {
        "nn_to_ucb": nn_to_ucb,
        "nn_to_ei": nn_to_ei,
        "ucb_to_ei": ucb_to_ei
    }

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(
        f"NN → UCB distance: "
        f"{nn_to_ucb:.4f}"
    )
    print(
        f"NN → EI distance:  "
        f"{nn_to_ei:.4f}"
    )
    print(
        f"UCB → EI distance: "
        f"{ucb_to_ei:.4f}"
    )

    print()

FUNCTION 1
NN → UCB distance: 0.4923
NN → EI distance:  0.4923
UCB → EI distance: 0.0000

FUNCTION 2
NN → UCB distance: 0.2851
NN → EI distance:  0.2908
UCB → EI distance: 0.0077

FUNCTION 3
NN → UCB distance: 0.5382
NN → EI distance:  0.2600
UCB → EI distance: 0.3695

FUNCTION 4
NN → UCB distance: 0.1355
NN → EI distance:  0.1504
UCB → EI distance: 0.0766

FUNCTION 5
NN → UCB distance: 0.1887
NN → EI distance:  0.1887
UCB → EI distance: 0.0000

FUNCTION 6
NN → UCB distance: 0.2208
NN → EI distance:  0.2055
UCB → EI distance: 0.1444

FUNCTION 7
NN → UCB distance: 0.3327
NN → EI distance:  0.3327
UCB → EI distance: 0.0000

FUNCTION 8
NN → UCB distance: 0.7737
NN → EI distance:  0.7737
UCB → EI distance: 0.0000



In [14]:
week_5_methods = {
    1: "EI / UCB agreement",
    2: "Expected Improvement",
    3: "Expected Improvement",
    4: "Expected Improvement",
    5: "Expected Improvement",
    6: "Neural Network",
    7: "EI / UCB agreement",
    8: "Neural Network"
}

historical_performance = []

for i in range(1, 9):

    previous_best = previous_bests[i]
    new_output = week_5_outputs[i]

    improvement = (
        new_output - previous_best
    )

    improved = (
        new_output > previous_best
    )

    historical_performance.append({
        "Function": i,
        "Previous method": week_5_methods[i],
        "Previous best": previous_best,
        "New output": new_output,
        "Improvement": improvement,
        "New best?": improved
    })

historical_performance_df = pd.DataFrame(
    historical_performance
)

historical_performance_df

,Function,Previous method,Previous best,New output,Improvement,New best?
0,1,EI / UCB agreement,7.710875e-16,-2.446767e-120,-7.710875e-16,False
1,2,Expected Improvement,6.112052e-01,7.469623e-01,1.357570e-01,True
2,3,Expected Improvement,-2.016691e-02,-1.645129e-02,3.715615e-03,True
3,4,Expected Improvement,-3.006447e-01,-6.383428e-01,-3.376982e-01,False
4,5,Expected Improvement,4.857450e+03,6.691932e+03,1.834482e+03,True
5,6,Neural Network,-4.585106e-01,-4.568735e-01,1.637079e-03,True
6,7,EI / UCB agreement,2.536939e+00,1.644736e+00,-8.922030e-01,False
7,8,Neural Network,9.911249e+00,9.742277e+00,-1.689723e-01,False


In [15]:
neighbourhood_results = {}

for i in range(1, 9):

    X_existing = (
        data[i]
        .drop(columns="output")
        .values
        .astype(float)
    )

    y_existing = (
        data[i]["output"]
        .values
        .astype(float)
    )

    candidates_to_check = {
        "UCB": gp_candidate_results[i]["ucb_point"],
        "EI": gp_candidate_results[i]["ei_point"],
        "NN": nn_candidate_results[i]["point"]
    }

    neighbourhood_results[i] = {}

    print("=" * 70)
    print(f"FUNCTION {i}")
    print("=" * 70)

    for method, point in candidates_to_check.items():

        # Calculate distance from candidate
        # to every existing observation
        distances = np.linalg.norm(
            X_existing - point,
            axis=1
        )

        # Find three nearest observations
        nearest_indices = np.argsort(
            distances
        )[:3]

        neighbourhood_results[i][method] = {
            "point": point,
            "distances": distances[nearest_indices],
            "outputs": y_existing[nearest_indices]
        }

        print(f"\n{method} candidate:")
        print(np.round(point, 6))

        print("Nearest observations:")

        for idx in nearest_indices:

            print(
                f"  Distance = {distances[idx]:.4f} "
                f"| Output = {y_existing[idx]:.6f}"
            )

    print()

FUNCTION 1

UCB candidate:
[0.773956 0.438878]
Nearest observations:
  Distance = 0.1807 | Output = 0.000000
  Distance = 0.1864 | Output = 0.000000
  Distance = 0.1873 | Output = -0.000000

EI candidate:
[0.773956 0.438878]
Nearest observations:
  Distance = 0.1807 | Output = 0.000000
  Distance = 0.1864 | Output = 0.000000
  Distance = 0.1873 | Output = -0.000000

NN candidate:
[9.98294e-01 7.09000e-04]
Nearest observations:
  Distance = 0.3077 | Output = 0.000000
  Distance = 0.3629 | Output = -0.000000
  Distance = 0.5927 | Output = 0.000000

FUNCTION 2

UCB candidate:
[0.732816 0.105087]
Nearest observations:
  Distance = 0.0385 | Output = 0.746962
  Distance = 0.0696 | Output = 0.538996
  Distance = 0.3343 | Output = 0.214965

EI candidate:
[0.729513 0.111999]
Nearest observations:
  Distance = 0.0309 | Output = 0.746962
  Distance = 0.0648 | Output = 0.538996
  Distance = 0.3277 | Output = 0.214965

NN candidate:
[0.998307 0.001142]
Nearest observations:
  Distance = 0.3164 | Ou

In [16]:
final_candidates = {
    1: {
        "point": gp_candidate_results[1]["ei_point"],
        "method": "EI / UCB agreement"
    },

    2: {
        "point": gp_candidate_results[2]["ei_point"],
        "method": "Expected Improvement"
    },

    3: {
        "point": gp_candidate_results[3]["ei_point"],
        "method": "Expected Improvement"
    },

    4: {
        "point": gp_candidate_results[4]["ei_point"],
        "method": "Expected Improvement"
    },

    5: {
        "point": nn_candidate_results[5]["point"],
        "method": "Neural Network"
    },

    6: {
        "point": gp_candidate_results[6]["ei_point"],
        "method": "Expected Improvement"
    },

    7: {
        "point": gp_candidate_results[7]["ei_point"],
        "method": "EI / UCB agreement"
    },

    8: {
        "point": nn_candidate_results[8]["point"],
        "method": "Neural Network"
    }
}

In [17]:
final_validation = {}

print("=" * 70)
print("FINAL WEEK 6 VALIDATION")
print("=" * 70)

for i in range(1, 9):

    point = np.array(
        final_candidates[i]["point"],
        dtype=float
    )

    method = final_candidates[i]["method"]

    X_existing = (
        data[i]
        .drop(columns="output")
        .values
        .astype(float)
    )

    # Check bounds
    in_bounds = np.all(
        (point >= 0) & (point <= 1)
    )

    # Check whether candidate duplicates
    # an existing observation
    duplicate = np.any(
        np.all(
            np.isclose(
                X_existing,
                point,
                atol=1e-8
            ),
            axis=1
        )
    )

    # Distance to nearest existing observation
    distances = np.linalg.norm(
        X_existing - point,
        axis=1
    )

    nearest_distance = np.min(
        distances
    )

    final_validation[i] = {
        "in_bounds": in_bounds,
        "duplicate": duplicate,
        "nearest_distance": nearest_distance
    }

    print()
    print(f"FUNCTION {i}")
    print(
        f"Selected point: "
        f"{np.round(point, 6)}"
    )
    print(
        f"Selection method: {method}"
    )
    print(
        f"In bounds: {in_bounds}"
    )
    print(
        f"Duplicate existing point: {duplicate}"
    )
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance:.6f}"
    )


# Overall validation
all_valid = all(
    result["in_bounds"]
    and not result["duplicate"]
    for result in final_validation.values()
)

print()
print("=" * 70)
print(
    f"ALL EIGHT POINTS VALID: {all_valid}"
)
print("=" * 70)

FINAL WEEK 6 VALIDATION

FUNCTION 1
Selected point: [0.773956 0.438878]
Selection method: EI / UCB agreement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.180671

FUNCTION 2
Selected point: [0.729513 0.111999]
Selection method: Expected Improvement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.030884

FUNCTION 3
Selected point: [0.30826  0.310001 0.479247]
Selection method: Expected Improvement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.140675

FUNCTION 4
Selected point: [0.329887 0.354827 0.355151 0.453774]
Selection method: Expected Improvement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.104897

FUNCTION 5
Selected point: [0.939869 0.992261 0.978815 0.999723]
Selection method: Neural Network
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.028637

FUNCTION 6
Selected point: [0.397519 0.393747 0.47144

In [18]:
print("WEEK 6 SUBMISSION POINTS")
print("=" * 70)

submission_points = {}

for i in range(1, 9):

    point = np.array(
        final_candidates[i]["point"],
        dtype=float
    )

    # Format every coordinate to 6 decimal places
    formatted_point = "-".join(
        f"{value:.6f}"
        for value in point
    )

    submission_points[i] = formatted_point

    print(
        f"Function {i}: {formatted_point}"
    )

WEEK 6 SUBMISSION POINTS
Function 1: 0.773956-0.438878
Function 2: 0.729513-0.111999
Function 3: 0.308260-0.310001-0.479247
Function 4: 0.329887-0.354827-0.355151-0.453774
Function 5: 0.939869-0.992261-0.978815-0.999723
Function 6: 0.397519-0.393747-0.471443-0.711380-0.165385
Function 7: 0.174885-0.324058-0.479146-0.269144-0.331135-0.614490
Function 8: 0.042609-0.051269-0.314729-0.127230-0.965221-0.369000-0.150066-0.389171
